# Grading buildings in post-earthquake photos

This notebook takes street-level photos, finds the buildings with SAM 3, crops each building and gives it a damage grade from A to F with the released model.

| Grade | Meaning |
|---|---|
| A | intact |
| B | intact, broken windows or minor damage |
| C | standing, damaged (cracks) |
| D | collapsed with voids, structure still recognisable |
| E | pancake collapse, stacked layers |
| F | debris |

The full definitions and borderline rules are in `docs/labelling_guideline.md`.

Requirements: a GPU (a Colab T4 is enough for one photo at a time), a Hugging Face account with access to `facebook/sam3` (request it on the model page), and the released grading weights.

In [ ]:
# On Colab: install the package from GitHub (takes a few minutes).
!pip -q install "insarag-triage[detect] @ git+https://github.com/leeelyehezkel/insarag-damage-triage"
!git clone -q https://github.com/leeelyehezkel/insarag-damage-triage repo 2>/dev/null || true

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageOps

from insarag_triage import detect as D
from insarag_triage.inference import load_model, predict

PHOTOS = Path('photos')        # folder with your photos (on Colab: upload them here)
OUT = Path('output')           # crops and results are written here
MODEL = 'leeely/insarag-triage-dinov3-vits16'   # Hugging Face repo id, or a local release folder
# the study's settings, or your own from notebook 00 (detect_custom.yaml)
cfg_file = Path('detect_custom.yaml') if Path('detect_custom.yaml').exists() else \
    Path('repo/configs/detect.yaml') if Path('repo').exists() else Path('configs/detect.yaml')
cfg = D.load_config(cfg_file)
print('SAM 3 prompts and thresholds:', cfg['prompts'])

## Log in to Hugging Face

SAM 3 is gated. Accept the terms on https://huggingface.co/facebook/sam3, then log in with a token (Settings → Access Tokens).

In [ ]:
from huggingface_hub import login
login()

## Load the models

SAM 3 finds the buildings with four text prompts, each with its own score threshold (see notebook 00 for how they were chosen). Boxes found by more than one prompt are merged (NMS, IoU 0.40). Boxes smaller than 4% of the photo width, 5% of its height or 0.4% of its area are dropped. The grading model is DINOv3 ViT-S/16 fine-tuned on 1,620 labelled crops, and its three training seeds are averaged.

In [ ]:
sam, sam_proc = D.load_sam3(cfg['model_id'])
grader = load_model(MODEL)
print('grading model:', grader.config['backbone'], grader.config['img_size'], 'px,', len(grader.models), 'seeds')

## Find and crop the buildings

Each building is cut out with 10% padding on every side. Crops with a short side under 96 px, or that are very dark, very bright or nearly uniform, are marked as unusable and not graded; in our data a person could not grade such crops reliably either. Aerial photos are not filtered here. In the study a small vision-language model removed them (see notebook 00).

In [ ]:
photos = sorted(p for p in PHOTOS.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'})
crops = pd.concat([D.crop_photo(p, sam, sam_proc, cfg, OUT / 'crops') for p in photos], ignore_index=True)
print(f"{len(photos)} photos, {len(crops)} buildings found, {int(crops.quality_ok.sum())} usable")
crops.groupby('prompt').size()

## Grade the crops

`grade` is the predicted grade. `expected_grade` is the model's mean grade on a 0–5 scale (A = 0, F = 5); it is useful for ranking buildings by severity. The `P(...)` columns are the probabilities.

In [ ]:
ok = crops[crops.quality_ok].reset_index(drop=True)
graded = ok.merge(predict(grader, ok['path']), on='path')
graded.to_csv(OUT / 'grades.csv', index=False)
graded[['photo', 'prompt', 'score', 'grade', 'expected_grade', 'P(C)', 'P(D)']].round(2)

## Look at the results

Each building is drawn with its box and grade on the photo.

In [ ]:
COLORS = dict(A='#2e7d32', B='#9e9d24', C='#f9a825', D='#ef6c00', E='#c62828', F='#4e342e')
for name, g in graded.groupby('photo'):
    img = ImageOps.exif_transpose(Image.open(PHOTOS / name)).convert('RGB')
    draw = ImageDraw.Draw(img)
    lw = max(2, img.width // 300)
    for _, r in g.iterrows():
        draw.rectangle([r.x, r.y, r.x + r.w, r.y + r.h], outline=COLORS[r.grade], width=lw)
        draw.text((r.x + lw, r.y + lw), r.grade, fill=COLORS[r.grade])
    plt.figure(figsize=(10, 7)); plt.imshow(img); plt.axis('off'); plt.title(name); plt.show()

## Notes

- The model was trained on photos from the February 2023 earthquakes in Türkiye. Accuracy on other events, building types and camera angles has not been tested yet.
- In our evaluations almost all errors were between neighbouring grades (A/B, B/C, C/D, E/F). Treat a single grade as an estimate, and use the probabilities when a decision depends on it.
- SAM 3 is used under the SAM License and the grading model under the DINOv3 License (see `licenses/`). Please cite the dataset and model as described in the README.